# MH²M with independently refined conormals

The oscillatory coefficient follows equation (61) of de Barros, Madureira and Valentin, arXiv:2404.16978v3, Section 8.2, with epsilon=1/14. Barros's 2022 LNCC dissertation supplies gamma=1.8 (Equation 4.4, printed page 62) and the quartic forcing (Equation 4.1, page 54). Its Figure 4 also specifies the crisscross fine connectivity and northwest-to-southeast macro diagonals. The dissertation uses epsilon=1/17; this notebook retains the version-3 article's epsilon=1/14. These source relationships do not identify every historical h/H convention.

This notebook runs a small interface-dimension check, then displays the multilevel numerical campaigns. The classical conforming references have their own refinement series and use the same physical data. The P1 controls use shared finite-element kernels; the P3 reference is independently assembled with DOLFINx/UFL. None is an exact solution.


In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm.mesh import FaceSpace, SkeletonSpace, TriangleMesh
from pymhm.mh2m import PressureTraceSpace, solve_mh2m

ROOT = Path.cwd()
if not (ROOT / "examples").exists():
    ROOT = ROOT.parent


## Separate interface spaces

The continuous pressure trace Gamma determines the global pressure-trace unknowns. The broken conormal Lambda determines the local Neumann maps. For the small unit-diffusion problem below, Gamma and the local P1 space remain fixed while Lambda is refined. The conormal is K grad(p)·n, opposite to the outward physical Darcy flux.

One P0 conormal per fine boundary edge can create an alternating kernel for continuous P1 pressures on an even boundary cycle. The tested pair uses at least two fine edges per conormal segment.

In [ ]:
mesh = TriangleMesh.unit_square(2)
gamma = PressureTraceSpace.uniform(mesh, 1, 1)
dimensions = []
with threadpool_limits(1):
    for segments in (1, 2):
        conormal = SkeletonSpace(mesh, tuple(FaceSpace.uniform(0, segments) for _ in mesh.faces))
        result = solve_mh2m(mesh, pressure_trace=gamma, flux_space=conormal,
                            degree=1, local_refinement=4, source=1.0)
        balance = float(np.max(np.abs(result.conservation_residuals())))
        assert balance < 2e-12
        dimensions.append((segments, result.trace_space.size, len(result.free_dofs)))
        print({"Lambda_segments": segments, "Gamma_total": result.trace_space.size,
               "Gamma_free": len(result.free_dofs), "macro_balance": balance})
assert dimensions[0][1:] == dimensions[1][1:]


## Independently refined classical P3 reference

DOLFINx/UFL assembles the same coefficient, source and boundary conditions on five SW–NE meshes. Consecutive differences use the finer field in each denominator. The final P3 n=512 flux increment is 0.0783701%; this is a refinement measurement, not an exact-error bound. Assembly orders 12 and 16 differ by about 2.39e-12 in relative flux. The archived-field energy agrees with a native UFL integral to about 1.99e-13 relative.

The 24 crisscross cases and 18 diagonal-mesh controls use this same reference for physical norms. Orders 8 and 10 are retained separately. The article's P1 n=128 curve remains unchanged in the published-profile overlays.


In [ ]:
cg3_root = ROOT / "examples/results/mh2m-heterogeneous/cg3"
cg3 = json.loads((cg3_root / "comparison.json").read_text())
cg3_controls = json.loads((cg3_root / "structured-comparison.json").read_text())
cg3_reference = cg3["reference_acquisitions"][-1]
assert len(cg3["reference_acquisitions"]) == 5 and len(cg3["increments"]) == 4
assert cg3_controls["reference_acquisition"] == cg3_reference
for acquisition in [*cg3["reference_acquisitions"],
                    cg3["assembly_quadrature_control"]["acquisition"]]:
    assert not acquisition["source_changed_during_run"]
    assert hashlib.sha256((cg3_root / acquisition["archive"]).read_bytes()).hexdigest() == acquisition["archive_sha256"]
for previous, current, row in zip(cg3["reference_acquisitions"][:-1],
                                  cg3["reference_acquisitions"][1:],
                                  cg3["increments"], strict=True):
    assert row["reference_archive"] == current["archive"]
    assert row["reference_sha256"] == current["archive_sha256"]
    assert row["other_archive"] == previous["archive"]
    assert row["other_sha256"] == previous["archive_sha256"]
    print(row["name"], row["norms"]["quadrature_10"])
print("Assembly quadrature control", cg3["assembly_quadrature_control"]["norms"])
print("P1 n=1024 versus P3 n=512", cg3["cg1_reference_comparison"]["norms"])
print(json.loads((cg3_root / "native-norm-verification.json").read_text()))


## Diagonal-mesh control series

These records use uniform triangular subdivisions with southwest-to-northeast diagonals. The pressure, broken gradient, raw physical flux and energy differences are integrated on a common subdivision resolving both triangle meshes. Pressure is not transferred to only one grid. The primary physical comparisons below use the independent continuous P3 n=512 field. The original P1 refinement series remains a separate reference-resolution control. Every norm uses its stated reference denominator; no percentage from one baseline is combined with another.


In [ ]:
record = json.loads((ROOT / "examples/results/mh2m-heterogeneous/comparison.json").read_text())
assert not record["source_changed_during_run"]
print(record["conventions"])
for row in record["references"]:
    if "increment" in row:
        print("Separate classical P1 control", row["resolution"], row["increment"])


In [ ]:
structured_norms = {row["archive"]: row for row in cg3_controls["cases"]}
assert len(structured_norms) == len(record["cases"]) == 18
for row in record["cases"]:
    comparison = structured_norms[row["archive"]]
    archive = ROOT / "examples/results/mh2m-heterogeneous" / row["archive"]
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == row["archive_sha256"]
    assert row["archive_sha256"] == comparison["archive_sha256"]
    print(row["method"], row["family"], row["macro_resolution"],
          row["local_refinement"], row["Lambda_segments"], row["global_dofs_free"],
          "versus P3 n=512", comparison["norms"]["quadrature_10"])


## Diagonal-mesh fields and interface enrichment

The fixed-Gamma series varies the conormal segments without increasing the global pressure-trace dimension. The diagonal-mesh spatial control uses n=31, local refinement 4, and two segments for Gamma and Lambda. This construction is distinct from the recovered crisscross connectivity below. One-sided profile segments and actual macro boundaries are retained. Raw flux maps are distinct from conormal moments and H(div) reconstructions.


In [ ]:
for name in ("refinement", "fixed-gamma", "profiles", "fields"):
    display(Image(filename=str(ROOT / f"docs/figures/mh2m-heterogeneous/{name}.png")))


## Recovered crisscross connectivity

Each macro has 2r² fine triangles and r,r,2r boundary edges. The local pairing with eight constant conormal segments per macroedge has rank 15 of 23 at r=4, and full rank 23 at r=8. A zero-mean Neumann map cannot invert the former pair. The recorded campaign uses admissible pairs and keeps the printed Figure 6 integer dimensions separate from its inconsistent subdivision label. The geometric maximum diameters are Hmax=sqrt(2)/n and hmax=1/(n*r), while the dissertation's Cartesian spacing is 1/n.

The Figure 5 configuration uses n=31, r=4 and two Gamma/Lambda segments. Both incident profile values are preserved. Its pressure profiles retain the printed P1 n=128 resolution; physical differences use the independently refined P3 n=512 field. Visual proximity alone does not establish historical reproduction.

The additional asinh view changes only the flux color normalization. Reference and MH²M share physical limits and linear widths; differences have their own limits, and all ticks retain physical values. Pressure, fields and norms are unchanged.

Figure-6 controls retain the printed global dimensions separately from the caption: four uniform subfaces give the lower-row dimensions, while eight conormal segments provide an explicit local enrichment at fixed Gamma. The upper starred r=4/sLambda=8 pair fails the local injectivity condition.


In [ ]:
cross_root = ROOT / "examples/results/mh2m-heterogeneous/crisscross"
cross = json.loads((cross_root / "comparison.json").read_text())
assert not cross["source_changed_during_run"]
cross_norms = {row["name"]: row for row in cg3["cases"]}
assert len(cross_norms) == len(cross["cases"]) == 24
print(cross["conventions"])
for row in cross["cases"]:
    assert hashlib.sha256((cross_root / row["archive"]).read_bytes()).hexdigest() == row["archive_sha256"]
    comparison = cross_norms[row["name"]]
    assert comparison["archive_sha256"] == row["archive_sha256"]
    assert comparison["reference_sha256"] == cg3_reference["archive_sha256"]
    print(row["name"], row["global_dofs_total"], row["global_dofs_free"],
          "versus P3 n=512", comparison["norms"]["quadrature_10"])
print(json.loads((cross_root / "space-compatibility.json").read_text())["rows"])


In [ ]:
for name in ("profiles", "fields", "fields-asinh", "refinement", "fixed-gamma",
             "figure-6-upper-profiles", "figure-6-lower-profiles",
             "published-figure-7-comparison", "published-figure-8-comparison"):
    path = ROOT / f"docs/figures/mh2m-heterogeneous/crisscross/{name}.png"
    display(Image(filename=str(path)))


The printed-profile comparisons use the original embedded raster colours and retain their visible line spans and one-pixel uncertainty. Occluded colours are not interpolated. Both incident numerical traces are shown without amplitude scaling. These profile comparisons are distinct from volume norms, and disagreement beyond raster uncertainty is retained.


In [ ]:
printed = json.loads((cross_root / "published-profile-comparison.json").read_text())
print(printed["comparison"])
for row in printed["rows"]:
    print(row["name"], row["estimates"])


The classical reference profile can also be checked independently of MH²M. The following controls retain the same coefficient, source and 128 Cartesian subdivisions, comparing northwest–southeast and crisscross connectivities at assembly orders 10 and 14. Crisscross has twice as many triangles; neither control replaces the independently refined reference series. Differences below are evaluated on visible raster columns and are not volume norms.

In [ ]:
geometry = json.loads((cross_root / "reference-geometry/comparison.json").read_text())
assert not geometry["source_changed_during_run"]
for row in geometry["cases"]:
    print(row["geometry"], row["quadrature_order"], row["dofs"], row["profiles"])
    if "previous_order_pressure_difference_l2_relative" in row:
        print("Quadrature pressure L2 change:", row["previous_order_pressure_difference_l2_relative"])

## Additional independent P1 verification

The independent P1 verification uses DOLFINx/UFL to assemble the six original n=32,...,1024 operators with the same physical Duffy-8 rule. It also includes a nonhomogeneous boundary patch. These checks distinguish the implemented classical field from identification of the historical figure.

The recorded n=128/130 and epsilon=1/14 or 1/17 controls, and nine fixed quadrature rules, do not reconcile all visible reference-curve columns in Figures 7 and 8. Raster envelopes include one pixel and the visible line thickness; no amplitude normalization is applied. These controls do not identify the unique historical data or geometry.

In [ ]:
native_p1 = json.loads((ROOT / "examples/results/mh2m-heterogeneous/cg1-native-verification.json").read_text())
assert not native_p1["source_changed_during_comparison"]
print(native_p1["scope"])
print(native_p1["nonhomogeneous_affine_patch"])
print(native_p1["fields"])
print(native_p1["documented_data_controls"])
print(native_p1["quadrature_controls"])